# 6.3 量化原理

## 1. 量化的定义（CANN 视角）

CANN 算子量化指对 Matmul 等**矩阵（Cube）类算子的输入 Tensor** 从高 bit 到低 bit 转换的计算过程，同时生成对应的量化参数 scale；低 bit 的 Cube 计算完成后，再通过 scale 把低 bit 数值转换回高 bit，保证整体计算结果与直接用高 bit 计算**近似等价**，同时有效提升计算效率。

数学上就是「缩放 + 舍入 + 截断」三步：

| 步骤 | 非对称量化 | 对称量化 |
|--|--|--|
| 量化 | q = round((r − Z) / S) | q = round(r / S) |
| 反量化 | r̂ = S·q + Z | r̂ = S·q |
| 定点区间 | [−128, 127] 或 [0, 255] | [−127, 127] |

其中 S 是 scale（步长），Z 是零点（zero point）。对称量化是非对称量化取 Z = 0 的特例；浮点格式（FP8/HiF8）天然带符号，通常用对称量化。

**scale 怎么定**：最常用 amax 法——对量化对象统计最大绝对值 `amax`，令 `S = amax / QMAX`（QMAX 是定点区间的最大值，如 E4M3 的 448、INT8 的 127）。这样动态范围恰好用满，代价是超出 amax 的离群值会被截断（clipping）。

## 2. 静态 vs 动态、四个易混术语

**静态/动态**按 scale 是离线生成还是在线生成分：

| | 静态量化 | 动态量化 |
|--|--|--|
| scale 来源 | 模型训练完成后预先确定，作为固定参数传入 | 推理过程中在线对 feature 统计并同步计算 |
| 优点 | 量化算子性能更好 | 更能适应数据变化，精度更高 |
| 缺点 | 精度略降 | 在线生成 scale 有额外开销 |
| 典型对象 | **weight**（不变） | **激活**（逐次输入不同）；训练为提精度也用动态 |

**四个术语**（结合后续计算的视角）：

| 术语 | 行为 | 位置 |
|--|--|--|
| 量化 Quant | 高精度 → 低 bit（如 BF16 → FP8） | 输入侧 |
| 伪量化 PseudoQuant | weight 按 INT8 存储、但按 FP16 计算，中间需插反量化算子 | 图优化阶段 |
| 反量化 AntiQuant / Dequant | AntiQuant：低 bit 输入转高 bit（INT8 → BF16），处理**输入**；Dequant：低 bit 计算结果（INT32）转高 bit，一般在 matmul 后的 **FixPipe 单元**处理 | 输入侧 / 输出侧 |
| 重量化 ReQuant | 低 bit 结果（INT32）转**低 bit**（INT8），同样在 FixPipe 处理，供下一层继续低 bit 计算 | 输出侧 |

## 3. 量化误差从哪里来

1. **舍入误差**：round 到最近可表示值，均匀噪声，通常无害；
2. **截断误差**：超出 [−amax, amax] 的离群值被 clip，**这是大模型量化的头号精度杀手**（离群通道的 amax 会把 S 拉大，其他通道的分辨率被牺牲）；
3. **scale 粒度误差**：一个 scale 罩住的范围越大、内部分布越不均，误差越大——这是 6.4 节各种量化粒度的动机；
4. **累加误差**：低 bit 乘法本身快，但累加必须高精度（FP32），否则误差沿 reduce 轴线性放大。

## 小测验

1. 写出对称与非对称量化的公式。浮点格式（FP8）为什么通常选对称？
2. amax 法求 scale 时，为什么离群值（outlier）是大模型量化的头号敌人？
3. AntiQuant 与 Dequant 都叫「反量化」，区别是什么？Dequant/ReQuant 一般发生在哪个硬件单元？
4. 静态量化为什么对 weight 更合适、动态量化为什么对激活更合适？各自付出什么代价？

In [ ]:
# 动手：用 numpy 实现对称/非对称量化，直观感受误差
import numpy as np

np.random.seed(0)
r = np.random.randn(8).astype(np.float32) * 0.5   # 一段「激活」数据
print('原始值   :', np.round(r, 3))

# ---------- 非对称量化到 int8 ----------
QMAX, QMIN = 127.0, -128.0
amax, amin = r.max(), r.min()
S = (amax - amin) / (QMAX - QMIN)     # 步长
Z = round(QMIN - amin / S)            # 零点（浮点 0 映射到的整数）
q_asym = np.clip(np.round(r / S + Z), QMIN, QMAX).astype(np.int8)
r_asym = (q_asym.astype(np.float32) - Z) * S
print('非对称q  :', q_asym)
print('非对称反量化误差:', np.round(r_asym - r, 4))

# ---------- 对称量化到 int8 ----------
amax_abs = np.abs(r).max()
S2 = amax_abs / QMAX
q_sym = np.clip(np.round(r / S2), QMIN, QMAX).astype(np.int8)
r_sym = q_sym.astype(np.float32) * S2
print('对称q    :', q_sym)
print('对称反量化误差  :', np.round(r_sym - r, 4))
print()
print(f'本例数据近似零对称，两者误差接近；若数据整体偏移（如全为正），')
print(f'对称量化会浪费一半定点区间——这正是 Z 存在的意义。')